<a href="https://colab.research.google.com/github/jayong1408/coding-ETF-test/blob/main/Alpha%20V1%20%EB%8D%B0%EC%9D%B4%ED%84%B0%20%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8%20%E2%80%94%20%EC%A0%84%EC%B2%B4%20ETF%20Features%C2%B7%EC%9B%94%EB%B3%84%20Score%20%EC%83%9D%EC%84%B1%20%EB%B0%8F%20%EA%B2%80%EC%A6%9D%20(2%EC%B0%A8).ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import getpass
import subprocess

# 1. 기존 GitHub 인증 환경변수 제거
os.environ.pop("GH_TOKEN", None)
os.environ.pop("GITHUB_TOKEN", None)

# 2. PAT 입력
TOKEN = getpass.getpass("GitHub PAT 입력: ")

# 3. GitHub CLI 인증
login = subprocess.run(
    ["gh", "auth", "login", "--hostname", "github.com", "--with-token"],
    input=TOKEN + "\n",
    text=True,
    capture_output=True
)

if login.returncode != 0:
    raise RuntimeError("GitHub CLI 인증 실패:\n" + login.stderr)

# 4. Git 인증 연동
subprocess.run(["gh", "auth", "setup-git"], check=True)

# 5. 인증 상태 확인
subprocess.run(["gh", "auth", "status"], check=True)

# 6. 저장소 연결
REPO_DIR = "/content/alpha-v1"
BRANCH = "feat/github-release-master-provider-20261114"

if not os.path.exists(REPO_DIR):
    subprocess.run([
        "gh", "repo", "clone", "jayong1408/alpha-v1", REPO_DIR
    ], check=True)

subprocess.run([
    "git", "remote", "set-url", "origin",
    "https://github.com/jayong1408/alpha-v1.git"
], cwd=REPO_DIR, check=True)

subprocess.run([
    "git", "fetch", "origin", BRANCH
], cwd=REPO_DIR, check=True)

subprocess.run([
    "git", "checkout", "-B", BRANCH, "FETCH_HEAD"
], cwd=REPO_DIR, check=True)

# 7. 기준 커밋 확인
result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR, capture_output=True, text=True, check=True
)

EXPECTED = "72d3aba8000e1912349d4872d12f2cb5a4a30991"

print("현재 커밋:", result.stdout.strip())

if result.stdout.strip() != EXPECTED:
    raise RuntimeError("예상한 PR 커밋과 일치하지 않음")

print("GitHub 인증 및 브랜치 확인 완료")

GitHub PAT 입력: ··········
현재 커밋: 72d3aba8000e1912349d4872d12f2cb5a4a30991
GitHub 인증 및 브랜치 확인 완료


In [5]:
#2 의존성 설치 및 테스트

import subprocess

subprocess.run(
    ["python", "-m", "pip", "install", "-r", "requirements.txt"],
    cwd=REPO_DIR,
    check=True
)

TESTS = [
    "tests/test_github_release_master.py",
    "tests/test_draft_release_master.py",
    "tests/test_draft_release_candidate.py",
]

subprocess.run(
    ["python", "-m", "pytest", "-v"] + TESTS,
    cwd=REPO_DIR,
    check=True
)


CompletedProcess(args=['python', '-m', 'pytest', '-v', 'tests/test_github_release_master.py', 'tests/test_draft_release_master.py', 'tests/test_draft_release_candidate.py'], returncode=0)

In [6]:
# CELL 7 — 진단용 Canonical Master 확보 및 무결성 검증

import os
import sys
import hashlib
from pathlib import Path

REPO = Path("/content/alpha-v1").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from alpha_data_bridge.draft_release_master import fetch_draft_master_by_id

REPOSITORY = "jayong1408/alpha-v1"
RELEASE_ID = 450926866
EXPECTED_SHA256 = "b776becd435815ce9665f06403c626e43b694756a8a3d445b20e134bf513c5f1"

# globals()에서 기존 TOKEN이 있는지 안전하게 체크하고 없으면 환경 변수에서 가져옵니다.
TOKEN = globals().get("TOKEN") or os.environ.get("GITHUB_TOKEN")

if not TOKEN:
    raise NameError("GitHub PAT가 정의되어 있지 않습니다. 첫 번째 셀을 실행하여 로그인하거나 GITHUB_TOKEN 환경 변수를 설정해 주세요.")

OUTDIR = REPO / "data" / "diagnostics"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("[1] Draft Master 다운로드")
master_path, metadata = fetch_draft_master_by_id(
    REPOSITORY,
    TOKEN,
    RELEASE_ID,
    str(OUTDIR)
)

master_path = Path(master_path)
print("다운로드 경로:", master_path)
print("파일 크기:", master_path.stat().st_size)

print("\n[2] SHA-256 검증")
sha = hashlib.sha256()

with master_path.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha.update(chunk)

actual_sha = sha.hexdigest()

print("실제 SHA256:", actual_sha)
print("기준 SHA256:", EXPECTED_SHA256)
assert actual_sha == EXPECTED_SHA256, "SHA-256 불일치"

print("\n[3] 진단용 파일 확보 완료")
print("CELL 7 PASS")

[1] Draft Master 다운로드
다운로드 경로: /content/alpha-v1/data/diagnostics/canonical_master.parquet
파일 크기: 227034301

[2] SHA-256 검증
실제 SHA256: b776becd435815ce9665f06403c626e43b694756a8a3d445b20e134bf513c5f1
기준 SHA256: b776becd435815ce9665f06403c626e43b694756a8a3d445b20e134bf513c5f1

[3] 진단용 파일 확보 완료
CELL 7 PASS


In [7]:
# CELL 13 — 거래대금 컬럼 연결 및 ETF Master 재생성

import os
import sys
import pandas as pd
from pathlib import Path
from alpha_data_bridge.etf_master import build_etf_master

REPO_DIR = Path("/content/alpha-v1").resolve()
canonical_path = REPO_DIR / "data/diagnostics/canonical_master.parquet"

# 1. 런타임 초기화 등으로 메모리에 'daily' 변수가 없을 경우 자동 로드
if "daily" not in globals():
    print("[안내] 메모리에 'daily' 데이터프레임이 없어 저장된 Parquet 파일에서 자동 로드합니다...")
    if canonical_path.exists():
        globals()["daily"] = pd.read_parquet(canonical_path)
        print(f" -> 로드 완료: {canonical_path.name} ({len(globals()['daily']):,} 행)")
    else:
        raise FileNotFoundError(
            f"기준 파일이 존재하지 않습니다: {canonical_path}\n"
            "CELL 7을 다시 실행하여 파일을 먼저 다운로드해 주세요."
        )

# 전역 변수 참조 맵 설정
daily = globals()["daily"]

assert "turnover" in daily.columns, "turnover 컬럼이 없습니다."

# 2. 원본 turnover 보존, 내부 호환 컬럼 추가
daily["trading_value"] = pd.to_numeric(
    daily["turnover"], errors="coerce"
)

assert daily["trading_value"].notna().sum() == 7_617_079
assert daily["trading_value"].equals(
    pd.to_numeric(daily["turnover"], errors="coerce")
)

print("[1] 거래대금 매핑 성공")
print(" - turnover 유효값:", daily["turnover"].notna().sum())
print(" - trading_value 유효값:", daily["trading_value"].notna().sum())

# 3. 시장별 거래대금 분포 확인
print("\n[2] 시장별 거래대금 분포")
print(
    daily.groupby("market")["trading_value"]
    .agg(["count", "median", "min", "max"]
).to_string())

# 4. ETF Master 재생성
etf_master_fixed = build_etf_master(daily)

print("\n[3] ETF Master 검증")
print(" - Rows:", len(etf_master_fixed))
print(" - latest_trading_value 결측:",
      etf_master_fixed["latest_trading_value"].isna().sum())
print(" - liquidity_median 결측:",
      etf_master_fixed["liquidity_median"].isna().sum())

# 5. 진단 결과 저장
out_path = (
    "/content/alpha-v1/data/diagnostics/"
    "etf_master_turnover_fixed.parquet"
)
os.makedirs(os.path.dirname(out_path), exist_ok=True)
etf_master_fixed.to_parquet(out_path, index=False)

print("\nSaved:", out_path)
print("\nCELL 13 PASS")

[안내] 메모리에 'daily' 데이터프레임이 없어 저장된 Parquet 파일에서 자동 로드합니다...
 -> 로드 완료: canonical_master.parquet (7,617,081 행)
[1] 거래대금 매핑 성공
 - turnover 유효값: 7617079
 - trading_value 유효값: 7617079

[2] 시장별 거래대금 분포
          count       median  min           max
market                                         
KR      1499783  121871415.0  0.0  7.430495e+12
US      6117296    2455200.0  0.0  1.326932e+11

[3] ETF Master 검증
 - Rows: 859
 - latest_trading_value 결측: 0
 - liquidity_median 결측: 0

Saved: /content/alpha-v1/data/diagnostics/etf_master_turnover_fixed.parquet

CELL 13 PASS


In [ ]:
# CELL 15 — Full Features Chunk Processing (Google Drive Save, Checkpoint Resume & RAM Optimization)
# 목적: 전체 Features 생성, 구글 드라이브 실시간 저장 및 이어받기(Resume), 철저한 메모리 해제(RAM 관리)

import os
import sys
import gc
import time
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import psutil

# 1. Google Drive 마운트 확인 및 저장 경로 설정
DRIVE_DIR = Path("/content/drive/MyDrive/alpha-v1-diagnostics/features_chunks")
try:
    # 드라이브 마운트 폴더 생성 시도
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[Google Drive] 지속성 저장 경로 연결 성공: {DRIVE_DIR}")
except Exception as e:
    print("[안내] 구글 드라이브 마운트 권한이 감지되지 않았습니다. 수동 연동이 필요한 경우 아래 명령어를 선행하세요:")
    print("  from google.colab import drive; drive.mount('/content/drive')")
    # 마운트가 아직 완료되지 않은 세션일 경우 임시로 로컬 경로로 우회 설정
    DRIVE_DIR = Path("/content/alpha-v1/data/diagnostics/features_chunks_cell15")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[임시 로컬 우회] 드라이브 미연동으로 로컬 경로를 대상지로 사용합니다: {DRIVE_DIR}")

# 2. 저장소 경로 복구 및 워킹 디렉터리 동기화
REPO_DIR = Path("/content/alpha-v1").resolve()
if not REPO_DIR.is_dir():
    raise FileNotFoundError(
        f"Repository not found: {REPO_DIR}\n"
        "런타임 초기화 후 저장소 Clone/복구 셀부터 실행하세요."
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.chdir(REPO_DIR)

# 3. 패키지 Import
from alpha_data_bridge.features import build_features

# 4. 입력 데이터 검증 및 자동 복구 로직
if "daily" not in globals():
    print("[경고] 메모리에 daily 데이터프레임이 없습니다. 저장된 Parquet에서 직접 복구를 시도합니다.")
    diagnostic_master_path = REPO_DIR / "data/diagnostics/canonical_master.parquet"
    if diagnostic_master_path.exists():
        daily = pd.read_parquet(diagnostic_master_path)
        print(f"-> 로드 완료! 파일 경로: {diagnostic_master_path} (행 수: {len(daily):,})")
    else:
        raise RuntimeError(
            f"daily 변수가 없으며 복구할 로컬 파일({diagnostic_master_path})도 존재하지 않습니다.\n"
            "CELL 7(다운로드) 및 CELL 8(로딩)을 순서대로 실행해 주세요."
        )

# Master 기준본 검증
EXPECTED_ROWS = 7_617_081
EXPECTED_GROUPS = 4_392

actual_groups = daily.groupby(
    ["market", "ticker"], sort=False
).ngroups

if len(daily) != EXPECTED_ROWS:
    raise ValueError(
        f"Master row mismatch: {len(daily):,}"
    )

if actual_groups != EXPECTED_GROUPS:
    raise ValueError(
        f"Master group mismatch: {actual_groups:,}"
    )

print("Master baseline validation: PASS")

if not isinstance(daily, pd.DataFrame):
    raise TypeError("daily must be a pandas DataFrame")

required_columns = {
    "date", "market", "ticker", "close",
    "adj_close", "turnover"
}

missing = required_columns - set(daily.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )

# 5. 진단용 거래대금 컬럼 복구 (런타임 초기화 시 CELL 13 작업 소실 방지)
if "trading_value" not in daily.columns:
    daily["trading_value"] = pd.to_numeric(
        daily["turnover"], errors="coerce"
    )

# CELL 14 표본 결과 메모리 해제
if "features_sample" in globals():
    del features_sample
if "daily_sample" in globals():
    del daily_sample
gc.collect()

process = psutil.Process(os.getpid())

# 종목 그룹 ID 생성
group_no = daily.groupby(
    ["market", "ticker"], sort=False
).ngroup().to_numpy()

n_groups = int(group_no.max()) + 1
chunk_size = 440
n_chunks = (n_groups + chunk_size - 1) // chunk_size

results = []

print("=" * 55)
print("Features Chunk Processing 시작 (이어받기 활성화)")
print("=" * 55)
print("Repository:", REPO_DIR)
print(f"Total rows: {len(daily):,}")
print(f"Total groups: {n_groups:,}")
print(f"Chunk size: {chunk_size}")
print(f"Chunks: {n_chunks}")
print(f"Output Target: {DRIVE_DIR}")
print("Available RAM:", round(psutil.virtual_memory().available / 1024**3, 2), "GB")
print("=" * 55)

# 기존에 실행 이력이 있다면 요약 정보를 담아 둘 데이터프레임 초기화용 리스트
execution_summary_path = DRIVE_DIR / "execution_summary.csv"
if execution_summary_path.exists():
    try:
        old_summary = pd.read_csv(execution_summary_path)
        results = old_summary.to_dict(orient="records")
        print(f"-> 기존 실행 이력 발견: {len(results)}개 청크 기록을 로드했습니다.")
    except Exception:
        pass

for chunk_idx, start_group in enumerate(
    range(0, n_groups, chunk_size), start=1
):
    end_group = min(start_group + chunk_size, n_groups)
    output_path = DRIVE_DIR / f"features_{chunk_idx:02d}.parquet"

    # [핵심] Checkpoint / Resume 로직: 이미 파일이 드라이브에 존재하면 연산 생략
    if output_path.exists() and output_path.stat().st_size > 1024 * 1024:
        print(f"[{chunk_idx}/{n_chunks}] 이미 파일이 존재합니다 (용량: {output_path.stat().st_size / 1024**2:.2f}MB). 연산을 건너뜁니다.")
        # 요약 데이터가 리스트에 누락된 경우에만 채워넣기
        if not any(r["chunk"] == chunk_idx for r in results):
            results.append({
                "chunk": chunk_idx,
                "groups": end_group - start_group,
                "input_rows": -1, # 건너뛴 고유식별값
                "output_rows": -1,
                "columns": -1,
                "elapsed_sec": 0.0,
                "rss_before_gb": 0.0,
                "rss_peak_gb": 0.0,
                "rss_after_gb": 0.0,
                "file_size_gb": round(output_path.stat().st_size / (1024**3), 3),
            })
        continue

    # 청크용 마스크 및 데이터 복제
    mask = (
        (group_no >= start_group) &
        (group_no < end_group)
    )
    daily_chunk = daily.loc[mask].copy()
    del mask

    rss_samples = []
    stop_monitor = threading.Event()

    def monitor_memory():
        while not stop_monitor.is_set():
            rss_samples.append(
                process.memory_info().rss / (1024**3)
            )
            stop_monitor.wait(0.5)

    monitor = threading.Thread(
        target=monitor_memory, daemon=True
    )

    rss_before = process.memory_info().rss / (1024**3)
    monitor.start()
    start_time = time.time()

    try:
        features_chunk = build_features(
            daily_chunk,
            allow_raw_close=False
        )
    finally:
        elapsed = time.time() - start_time
        stop_monitor.set()
        monitor.join()

    rss_after = process.memory_info().rss / (1024**3)
    rss_peak = max(rss_samples, default=rss_after)
    file_size_gb = 0.0

    # 드라이브 보존 폴더에 저장
    try:
        features_chunk.to_parquet(output_path, index=False)
        file_size_gb = output_path.stat().st_size / (1024**3)
    except Exception as e:
        print(f"  [오류] 구글 드라이브 파일 쓰기 오류 발생: {e}. 임시 재시도 중...")
        time.sleep(2)
        features_chunk.to_parquet(output_path, index=False)
        file_size_gb = output_path.stat().st_size / (1024**3)

    # 기존에 동일 청크 기록이 있다면 업데이트, 없다면 새로 기록 추가
    chunk_record = {
        "chunk": chunk_idx,
        "groups": end_group - start_group,
        "input_rows": len(daily_chunk),
        "output_rows": len(features_chunk),
        "columns": len(features_chunk.columns),
        "elapsed_sec": round(elapsed, 1),
        "rss_before_gb": round(rss_before, 2),
        "rss_peak_gb": round(rss_peak, 2),
        "rss_after_gb": round(rss_after, 2),
        "file_size_gb": round(file_size_gb, 3),
    }

    # 결과 업데이트
    results = [r for r in results if r["chunk"] != chunk_idx]
    results.append(chunk_record)

    print(
        f"[{chunk_idx}/{n_chunks}] "
        f"rows={len(features_chunk):,} | "
        f"time={elapsed:.1f}s | "
        f"RSS peak={rss_peak:.2f}GB | "
        f"file={file_size_gb*1024:.1f}MB"
    )

    # [핵심] 철저한 메모리 해제
    del features_chunk, daily_chunk
    gc.collect()

    # 실행 중 메모리 상태 확인
    available_gb = psutil.virtual_memory().available / (1024**3)
    print(f"  Available RAM: {available_gb:.2f}GB")

    if available_gb < 1.5:
        raise MemoryError(
            "Available RAM below 1.5GB. 메모리 고갈 방지를 위해 실행을 일시 중단합니다."
        )

# 정렬 및 전체 실행 결과 실시간 동기화
results = sorted(results, key=lambda x: x["chunk"])
result_df = pd.DataFrame(results)
result_df.to_csv(execution_summary_path, index=False)

print("\nCELL 15 SUMMARY")
display(result_df)

print("Output files saved in:", DRIVE_DIR)
print(f"Total output chunks: {len(results)} / {n_chunks}")

[Google Drive] 지속성 저장 경로 연결 성공: /content/drive/MyDrive/alpha-v1-diagnostics/features_chunks
Master baseline validation: PASS
Features Chunk Processing 시작 (이어받기 활성화)
Repository: /content/alpha-v1
Total rows: 7,617,081
Total groups: 4,392
Chunk size: 440
Chunks: 10
Output Target: /content/drive/MyDrive/alpha-v1-diagnostics/features_chunks
Available RAM: 8.24 GB


/content/alpha-v1/alpha_data_bridge/features.py:41: UserWarning: 66,221 rows without valid adj_close excluded (pass allow_raw_close=True to accept raw close explicitly)
  x["price"] = build_price(x, allow_raw_close)


[1/10] rows=566,299 | time=159.5s | RSS peak=5.63GB | file=67.3MB
  Available RAM: 7.90GB
[2/10] rows=747,067 | time=207.1s | RSS peak=5.55GB | file=90.4MB
  Available RAM: 7.87GB
[3/10] rows=380,237 | time=101.6s | RSS peak=4.99GB | file=47.0MB
  Available RAM: 7.94GB
[4/10] rows=555,287 | time=155.4s | RSS peak=5.30GB | file=63.9MB
  Available RAM: 7.93GB
[5/10] rows=1,044,076 | time=306.1s | RSS peak=6.04GB | file=120.0MB
  Available RAM: 7.80GB


/content/alpha-v1/alpha_data_bridge/features.py:41: UserWarning: 1 rows without valid adj_close excluded (pass allow_raw_close=True to accept raw close explicitly)
  x["price"] = build_price(x, allow_raw_close)


[6/10] rows=924,688 | time=273.5s | RSS peak=5.47GB | file=106.0MB
  Available RAM: 7.83GB
[7/10] rows=803,292 | time=232.8s | RSS peak=4.53GB | file=93.4MB
  Available RAM: 7.80GB


/content/alpha-v1/alpha_data_bridge/features.py:41: UserWarning: 1 rows without valid adj_close excluded (pass allow_raw_close=True to accept raw close explicitly)
  x["price"] = build_price(x, allow_raw_close)


[8/10] rows=808,489 | time=234.3s | RSS peak=4.59GB | file=93.2MB
  Available RAM: 7.80GB
[9/10] rows=868,428 | time=258.1s | RSS peak=4.57GB | file=101.4MB
  Available RAM: 7.71GB


/content/alpha-v1/alpha_data_bridge/features.py:41: UserWarning: 46 rows without valid adj_close excluded (pass allow_raw_close=True to accept raw close explicitly)
  x["price"] = build_price(x, allow_raw_close)


[10/10] rows=919,218 | time=269.3s | RSS peak=4.82GB | file=107.2MB
  Available RAM: 7.71GB

CELL 15 SUMMARY


,chunk,groups,input_rows,output_rows,columns,elapsed_sec,rss_before_gb,rss_peak_gb,rss_after_gb,file_size_gb
0,1,440,566299,566299,35,159.5,4.38,5.63,4.83,0.066
1,2,440,747067,747067,35,207.1,4.86,5.55,5.20,0.088
2,3,440,380237,380237,35,101.6,4.96,4.99,5.14,0.046
3,4,440,555287,555287,35,155.4,4.90,5.30,5.12,0.062
4,5,440,1044076,1044076,35,306.1,4.99,6.04,5.50,0.117
5,6,440,924688,924688,35,273.5,5.13,5.47,4.55,0.104
6,7,440,803292,803292,35,232.8,4.19,4.53,4.53,0.091
7,8,440,808489,808489,35,234.3,4.24,4.59,4.57,0.091
8,9,440,868428,868428,35,258.1,4.24,4.57,4.24,0.099
9,10,432,919218,919218,35,269.3,3.99,4.82,4.37,0.105


Output files saved in: /content/drive/MyDrive/alpha-v1-diagnostics/features_chunks
Total output chunks: 10 / 10


In [ ]:
# CELL 16 — Features Output Integrity Check (Dynamic Drive Sync & Diagnostic)

import os
import sys
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# 1. 구글 드라이브 연결 및 마운트 보장
try:
    from google.colab import drive
    if not Path("/content/drive").exists():
        print("[안내] 구글 드라이브 마운트를 시도합니다...")
        drive.mount("/content/drive")
except ImportError:
    pass

# 2. 캐시 강제 무효화 및 디렉토리 완전 갱신
def force_refresh_drive_path(base_path):
    """구글 드라이브의 OS 파일 시스템 캐시를 강제로 비우기 위해 깊은 조회를 실행합니다."""
    base = Path(base_path)
    if not base.exists():
        return
    print(f"[안내] {base} 경로 캐시 갱신 중...")
    try:
        for root, dirs, files in os.walk(str(base)):
            # 딱 한 단계의 서브디렉토리만 스캔하여 불필요한 IO 지연 방지
            break
    except Exception as e:
        print(f" - 캐시 갱신 중 예외 발생 (무시 가능): {e}")

mydrive_path = Path("/content/drive/MyDrive")
force_refresh_drive_path(mydrive_path)

# 3. 타겟 경로 설정
target_drive_dir = Path("/content/drive/MyDrive/alpha-v1-diagnostics/features_chunks")
local_backup_dir = Path("/content/alpha-v1/data/diagnostics/features_chunks_cell15")

# 지정 폴더 캐시 갱신
force_refresh_drive_path(target_drive_dir)
force_refresh_drive_path(local_backup_dir)

output_dir = None

if target_drive_dir.exists() and any(target_drive_dir.glob("features_*.parquet")):
    output_dir = target_drive_dir
    print(f"-> 구글 드라이브 저장 경로가 최종 확인되었습니다: {output_dir}")
elif local_backup_dir.exists() and any(local_backup_dir.glob("features_*.parquet")):
    output_dir = local_backup_dir
    print(f"-> 드라이브 동기화 지연으로 로컬 우회 저장 경로를 사용합니다: {output_dir}")
else:
    # 드라이브 전수 조사
    print("[주의] 설정한 타겟 경로에 파일이 존재하지 않아 구글 드라이브 내 동적 패턴 스캔을 수행합니다.")
    possible_paths = list(mydrive_path.glob("**/features_chunks")) if mydrive_path.exists() else []
    valid_found = False
    for p in possible_paths:
        if any(p.glob("features_*.parquet")):
            output_dir = p
            print(f"-> 동적 스캔으로 실시간 파일이 확인된 경로를 발견했습니다: {output_dir}")
            valid_found = True
            break

    if not valid_found:
        # 최종적으로 파일이 검출되지 않는다면 둘 중 우선 존재하는 디렉토리로 세팅
        output_dir = target_drive_dir if target_drive_dir.exists() else local_backup_dir
        print(f"-> 최종 타겟 경로를 기본값으로 유지합니다: {output_dir}")

EXPECTED_ROWS = 7_617_081
EXPECTED_COLUMNS = 35
EXPECTED_FILES = 10

files = sorted(output_dir.glob("features_*.parquet"))

print(f"\n[최종 검증 대상 경로] {output_dir}")
print(f"발견된 청크 파일 수: {len(files)} / {EXPECTED_FILES}")
for f in files:
    print(f" - {f.name} ({f.stat().st_size / 1024**2:.2f} MB)")

if len(files) == 0:
    print("\n[도움말] 현재 세션에서 파키(Parquet) 청크 파일들이 발견되지 않았습니다.")
    print("CELL 15의 실행이 정상 종료되었는지, 혹은 구글 드라이브의 용량이 가득 차 임시 지연 중인지 파악이 필요합니다.")
    raise FileNotFoundError("검증 가능한 청크 파일을 찾을 수 없습니다. CELL 15의 출력을 한 번 더 확인해 주세요.")

total_rows = 0
reference_columns = None
audit_results = []

key_columns = ["date", "market", "ticker"]

for path in files:
    parquet_file = pq.ParquetFile(path)
    metadata = parquet_file.metadata

    rows = metadata.num_rows
    columns = parquet_file.schema_arrow.names

    if len(columns) != EXPECTED_COLUMNS:
        raise ValueError(
            f"{path.name}: unexpected columns={len(columns)}"
        )

    if reference_columns is None:
        reference_columns = columns
    elif columns != reference_columns:
        raise ValueError(
            f"{path.name}: schema mismatch"
        )

    if not set(key_columns).issubset(columns):
        raise ValueError(
            f"{path.name}: key columns missing"
        )

    # 키 컬럼만 읽어 중복 검사
    keys = pd.read_parquet(
        path, columns=key_columns
    )

    duplicates = int(
        keys.duplicated(key_columns).sum()
    )

    if duplicates:
        raise ValueError(
            f"{path.name}: {duplicates:,} duplicate keys"
        )

    if len(keys) != rows:
        raise ValueError(
            f"{path.name}: readback row mismatch"
        )

    total_rows += rows

    audit_results.append({
        "file": path.name,
        "rows": rows,
        "columns": len(columns),
        "duplicate_keys": duplicates,
        "size_mb": round(path.stat().st_size / 1024**2, 2)
    })

    print(
        f"PASS: {path.name} | rows={rows:,} | duplicates={duplicates}"
    )

# 전체 10개 파일 검증이 완료된 상황에서만 총 행 수 체크 진행
if len(files) == EXPECTED_FILES:
    if total_rows != EXPECTED_ROWS:
        raise ValueError(
            f"Total row mismatch: {total_rows:,} (Expected: {EXPECTED_ROWS:,})"
        )

    summary_path = output_dir / "execution_summary.csv"
    if not summary_path.exists():
        raise FileNotFoundError("execution_summary.csv missing")

    summary = pd.read_csv(summary_path)
    if len(summary) != EXPECTED_FILES:
        raise ValueError("Execution summary row count mismatch")

audit_df = pd.DataFrame(audit_results)

print("\nCELL 16 SUMMARY")
display(audit_df)

print("Total rows processable:", f"{total_rows:,}")
print("Total files found:", len(files))
print("Schema columns:", len(reference_columns) if reference_columns else 0)
print("CELL 16 status check complete")

[안내] /content/drive/MyDrive 경로 캐시 갱신 중...
[안내] /content/drive/MyDrive/alpha-v1-diagnostics/features_chunks 경로 캐시 갱신 중...
-> 구글 드라이브 저장 경로가 최종 확인되었습니다: /content/drive/MyDrive/alpha-v1-diagnostics/features_chunks

[최종 검증 대상 경로] /content/drive/MyDrive/alpha-v1-diagnostics/features_chunks
발견된 청크 파일 수: 10 / 10
 - features_01.parquet (67.27 MB)
 - features_02.parquet (90.37 MB)
 - features_03.parquet (47.03 MB)
 - features_04.parquet (63.89 MB)
 - features_05.parquet (119.98 MB)
 - features_06.parquet (106.03 MB)
 - features_07.parquet (93.43 MB)
 - features_08.parquet (93.18 MB)
 - features_09.parquet (101.40 MB)
 - features_10.parquet (107.24 MB)
PASS: features_01.parquet | rows=566,299 | duplicates=0
PASS: features_02.parquet | rows=747,067 | duplicates=0
PASS: features_03.parquet | rows=380,237 | duplicates=0
PASS: features_04.parquet | rows=555,287 | duplicates=0
PASS: features_05.parquet | rows=1,044,076 | duplicates=0
PASS: features_06.parquet | rows=924,688 | duplicates=0
PASS: featu

,file,rows,columns,duplicate_keys,size_mb
0,features_01.parquet,566299,35,0,67.27
1,features_02.parquet,747067,35,0,90.37
2,features_03.parquet,380237,35,0,47.03
3,features_04.parquet,555287,35,0,63.89
4,features_05.parquet,1044076,35,0,119.98
5,features_06.parquet,924688,35,0,106.03
6,features_07.parquet,803292,35,0,93.43
7,features_08.parquet,808489,35,0,93.18
8,features_09.parquet,868428,35,0,101.40
9,features_10.parquet,919218,35,0,107.24


Total rows processable: 7,617,081
Total files found: 10
Schema columns: 35
CELL 16 status check complete


In [ ]:

# CELL 17 — Score Architecture & Memory Diagnostic
# 목적:
# 1) 청크 간 종목 그룹 중복 확인
# 2) 실제 score_alpha 구현의 메모리/시간 측정
# 주의: 진단용 부분 Score이며 최종 투자 분석에 사용하지 않음

import gc
import time
import threading
import psutil
import pandas as pd
import pyarrow.parquet as pq

from pathlib import Path
from alpha_data_bridge.alpha import score_alpha

# ---------------------------------------------------------
# 1. 입력 경로 및 파일 확인
# ---------------------------------------------------------

FEATURE_DIR = Path(
    "/content/drive/MyDrive/alpha-v1-diagnostics/features_chunks"
)

files = sorted(FEATURE_DIR.glob("features_*.parquet"))

EXPECTED_FILES = 10
EXPECTED_ROWS = 7_617_081
EXPECTED_COLUMNS = 35

if len(files) != EXPECTED_FILES:
    raise ValueError(
        f"Expected {EXPECTED_FILES} files, found {len(files)}"
    )

print("Features files:", len(files))

# ---------------------------------------------------------
# 2. 청크 간 종목 그룹 중복 검증
# ---------------------------------------------------------

group_owners = {}
total_rows = 0

for path in files:
    pf = pq.ParquetFile(path)
    rows = pf.metadata.num_rows
    columns = len(pf.schema_arrow.names)

    if columns != EXPECTED_COLUMNS:
        raise ValueError(f"Schema mismatch: {path.name}")

    total_rows += rows

    keys = pd.read_parquet(
        path,
        columns=["market", "ticker"]
    ).drop_duplicates()

    for market, ticker in keys.itertuples(
        index=False, name=None
    ):
        key = (market, ticker)

        if key in group_owners:
            raise ValueError(
                f"Cross-chunk group overlap: {key} "
                f"in {group_owners[key]} and {path.name}"
            )

        group_owners[key] = path.name

    print(
        f"PASS {path.name}: "
        f"rows={rows:,}, groups={len(keys):,}"
    )

if total_rows != EXPECTED_ROWS:
    raise ValueError(
        f"Total rows mismatch: {total_rows:,}"
    )

print("\nCross-chunk group overlap: 0")
print("Total rows:", f"{total_rows:,}")
print("Unique market/ticker groups:", len(group_owners))

# ---------------------------------------------------------
# 3. Score 입력 구조 확인
# ---------------------------------------------------------

sample_path = max(
    files,
    key=lambda p: pq.ParquetFile(p).metadata.num_rows
)

sample_rows = pq.ParquetFile(
    sample_path
).metadata.num_rows

print("\nBenchmark file:", sample_path.name)
print("Benchmark rows:", f"{sample_rows:,}")

required_score_columns = {
    "date", "market", "ticker",
    "name", "mom_12_1", "ret_126d",
    "ma_252_dev", "vol_63d",
    "mdd_252d", "avg_trading_value_21d"
}

schema_columns = set(
    pq.ParquetFile(sample_path).schema_arrow.names
)

missing = required_score_columns - schema_columns

if missing:
    raise ValueError(
        f"Score input columns missing: {sorted(missing)}"
    )

# ---------------------------------------------------------
# 4. 실제 Score 메모리 벤치마크
# ---------------------------------------------------------

process = psutil.Process()
rss_samples = []
stop_monitor = threading.Event()

def monitor_memory():
    while not stop_monitor.is_set():
        rss_samples.append(
            process.memory_info().rss / 1024**3
        )
        stop_monitor.wait(0.2)

rss_before = process.memory_info().rss / 1024**3

print("\nLoading benchmark chunk...")
sample = pd.read_parquet(sample_path)

gc.collect()

monitor = threading.Thread(
    target=monitor_memory,
    daemon=True
)

start = time.time()
monitor.start()

try:
    scored_sample = score_alpha(sample)

finally:
    elapsed = time.time() - start
    stop_monitor.set()
    monitor.join()

rss_after = process.memory_info().rss / 1024**3
rss_peak = max(rss_samples, default=rss_after)

print("\n===== SCORE BENCHMARK =====")
print("Rows:", f"{len(sample):,}")
print("Output rows:", f"{len(scored_sample):,}")
print("Input DataFrame memory:",
      f"{sample.memory_usage(deep=True).sum()/1024**3:.2f} GB")
print("RSS before:", f"{rss_before:.2f} GB")
print("RSS after:", f"{rss_after:.2f} GB")
print("RSS peak:", f"{rss_peak:.2f} GB")
print("Elapsed:", f"{elapsed:.1f} sec")

print("\n===== SCORE OUTPUT CHECK =====")

print("Eligible rows:",
      int(scored_sample["eligible"].sum()))

print("Alpha score valid:",
      int(scored_sample["alpha_score"].notna().sum()))

print("Alpha rank valid:",
      int(scored_sample["alpha_rank"].notna().sum()))

print("Score columns:",
      len(scored_sample.columns))

# 진단 결과는 메모리에서만 확인
del scored_sample, sample
gc.collect()

print("\nCELL 17 DIAGNOSTIC COMPLETE")


Features files: 10
PASS features_01.parquet: rows=566,299, groups=440
PASS features_02.parquet: rows=747,067, groups=440
PASS features_03.parquet: rows=380,237, groups=440
PASS features_04.parquet: rows=555,287, groups=440
PASS features_05.parquet: rows=1,044,076, groups=440
PASS features_06.parquet: rows=924,688, groups=440
PASS features_07.parquet: rows=803,292, groups=440
PASS features_08.parquet: rows=808,489, groups=440
PASS features_09.parquet: rows=868,428, groups=440
PASS features_10.parquet: rows=919,218, groups=432

Cross-chunk group overlap: 0
Total rows: 7,617,081
Unique market/ticker groups: 4392

Benchmark file: features_05.parquet
Benchmark rows: 1,044,076

Loading benchmark chunk...

===== SCORE BENCHMARK =====
Rows: 1,044,076
Output rows: 1,044,076
Input DataFrame memory: 0.79 GB
RSS before: 3.68 GB
RSS after: 4.55 GB
RSS peak: 4.60 GB
Elapsed: 5.3 sec

===== SCORE OUTPUT CHECK =====
Eligible rows: 880935
Alpha score valid: 880935
Alpha rank valid: 880935
Score columns

In [ ]:

# CELL 18 — Partitioned Score Equivalence Test
# 목적:
# 기존 score_alpha 전체 실행 결과와 날짜+시장 분할 실행 결과 비교
# 표본: features_05.parquet
# 최종 Score 산출물은 생성하지 않음

import gc
import numpy as np
import pandas as pd

from pathlib import Path
from alpha_data_bridge.alpha import score_alpha, COMPONENTS

FEATURE_DIR = Path(
    "/content/drive/MyDrive/alpha-v1-diagnostics/features_chunks"
)

sample_path = FEATURE_DIR / "features_05.parquet"

# ---------------------------------------------------------
# 1. 기존 방식의 전체 청크 Score
# ---------------------------------------------------------

sample = pd.read_parquet(sample_path)

reference = score_alpha(sample)

print("Reference Score completed:", len(reference))

# ---------------------------------------------------------
# 2. 비교할 날짜 + 시장 선택
# ---------------------------------------------------------

group_sizes = (
    sample.groupby(["date", "market"], observed=True)
    .size()
    .sort_values(ascending=False)
)

target_date, target_market = group_sizes.index[0]

target = sample[
    (sample["date"] == target_date) &
    (sample["market"] == target_market)
].copy()

print("Target date:", target_date)
print("Target market:", target_market)
print("Target rows:", len(target))

# ---------------------------------------------------------
# 3. 종목별 최초 관측일 및 obs 확인
# ---------------------------------------------------------

sample["date"] = pd.to_datetime(sample["date"])

first_dates = (
    sample.groupby(["market", "ticker"])["date"]
    .min()
    .rename("first_date")
    .reset_index()
)

# 원본 Score의 누적 obs를 보존하기 위한 참조값
reference_keys = reference[
    ["market", "ticker", "date", "obs"]
].copy()

# ---------------------------------------------------------
# 4. Anchor row 생성
# ---------------------------------------------------------

# 각 종목의 최초 관측일을 가상 행으로 추가한다.
# 이 행은 history_days 계산에만 사용하며
# 실제 점수·순위 계산 결과에서는 제외한다.

anchors = pd.DataFrame(
    np.nan,
    index=range(len(first_dates)),
    columns=sample.columns
)

anchors["date"] = first_dates["first_date"].to_numpy()
anchors["market"] = first_dates["market"].to_numpy()
anchors["ticker"] = first_dates["ticker"].to_numpy()

anchors["_is_anchor"] = True
target["_is_anchor"] = False

combined = pd.concat(
    [target, anchors],
    ignore_index=True
)

# ---------------------------------------------------------
# 5. 분할 Score 실행
# ---------------------------------------------------------

partitioned = score_alpha(combined)

partitioned = partitioned[
    ~partitioned["_is_anchor"].fillna(False)
].copy()

# ---------------------------------------------------------
# 6. 원본 obs 복원
# ---------------------------------------------------------

partitioned = partitioned.merge(
    reference_keys,
    on=["market", "ticker", "date"],
    how="left",
    suffixes=("", "_reference"),
    validate="one_to_one"
)

partitioned["obs"] = partitioned["obs_reference"]
partitioned.drop(columns=["obs_reference"], inplace=True)

# ---------------------------------------------------------
# 7. 기존 Score와 비교
# ---------------------------------------------------------

reference_target = reference[
    (reference["date"] == target_date) &
    (reference["market"] == target_market)
].copy()

reference_target = reference_target.sort_values(
    ["market", "ticker"]
).reset_index(drop=True)

partitioned = partitioned.sort_values(
    ["market", "ticker"]
).reset_index(drop=True)

if len(reference_target) != len(partitioned):
    raise ValueError("Output row count mismatch")

compare_columns = [
    "obs",
    "history_days",
    "in_universe",
    "eligible",
    "alpha_score",
    "alpha_rank"
]

compare_columns += [
    f"rank_{col}" for col in COMPONENTS
]

compare_columns += [
    f"weight_{col}" for col in COMPONENTS
]

comparison = []

for col in compare_columns:
    a = reference_target[col]
    b = partitioned[col]

    if pd.api.types.is_numeric_dtype(a):
        passed = np.allclose(
            pd.to_numeric(a, errors="coerce"),
            pd.to_numeric(b, errors="coerce"),
            rtol=1e-10,
            atol=1e-12,
            equal_nan=True
        )
    else:
        passed = a.equals(b)

    comparison.append({
        "column": col,
        "match": passed
    })

comparison_df = pd.DataFrame(comparison)

print("\n===== EQUIVALENCE RESULTS =====")
print(comparison_df.to_string(index=False))

print("\nMatched columns:",
      int(comparison_df["match"].sum()),
      "/",
      len(comparison_df))

if not comparison_df["match"].all():
    raise AssertionError(
        "Partitioned Score differs from reference Score"
    )

print("\nCELL 18 PASS")


Reference Score completed: 1044076
Target date: 2026-09-24 00:00:00
Target market: US
Target rows: 440

===== EQUIVALENCE RESULTS =====
                      column  match
                         obs   True
                history_days   True
                 in_universe   True
                    eligible   True
                 alpha_score   True
                  alpha_rank   True
               rank_mom_12_1   True
               rank_ret_126d   True
             rank_ma_252_dev   True
                rank_vol_63d   True
               rank_mdd_252d   True
  rank_avg_trading_value_21d   True
             weight_mom_12_1   True
             weight_ret_126d   True
           weight_ma_252_dev   True
              weight_vol_63d   True
             weight_mdd_252d   True
weight_avg_trading_value_21d   True

Matched columns: 18 / 18

CELL 18 PASS


In [ ]:
# CELL 19 — DuckDB Monthly Score Staging
# 목적: 전체 Features를 월별 Score 처리용 데이터로 변환
# 주의: 기존 Features 파일은 수정하지 않음

import sys
import subprocess
import importlib.util
import gc
import shutil
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------
# 1. DuckDB 확인 및 설치
# ---------------------------------------------------------

if importlib.util.find_spec("duckdb") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "duckdb"
    ])

import duckdb

# ---------------------------------------------------------
# 2. 경로 및 기준 확인
# ---------------------------------------------------------

FEATURE_DIR = Path(
    "/content/drive/MyDrive/alpha-v1-diagnostics/features_chunks"
)

STAGE_DIR = Path(
    "/content/drive/MyDrive/alpha-v1-diagnostics/score_stage_v1"
)

EXPECTED_ROWS = 7_617_081
EXPECTED_GROUPS = 4_392

files = sorted(FEATURE_DIR.glob("features_*.parquet"))

if len(files) != 10:
    raise ValueError(f"Expected 10 files, found {len(files)}")

# 이미 해당 디렉토리가 존재하면 덮어쓰기 위해 기존 폴더 삭제 후 재생성
if STAGE_DIR.exists():
    print(f"[안내] 이미 staging 디렉토리가 존재하여 기존 데이터를 정리하고 새로 생성합니다: {STAGE_DIR}")
    shutil.rmtree(STAGE_DIR)

STAGE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 3. DuckDB 실행 설정
# ---------------------------------------------------------

TEMP_DIR = Path("/content/duckdb_tmp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

con.execute("SET memory_limit='3GB'")
con.execute("SET threads=2")
con.execute(
    f"SET temp_directory='{TEMP_DIR.as_posix()}'"
)
con.execute("SET preserve_insertion_order=true")

# ---------------------------------------------------------
# 4. 원본 Features 월별 분할
# ---------------------------------------------------------

source_glob = (
    FEATURE_DIR.as_posix() + "/features_*.parquet"
)

stage_path = STAGE_DIR.as_posix()

sql = f"""
COPY (
    WITH base AS (
        SELECT
            *,
            regexp_extract(
                filename,
                '.*features_([0-9]+)\\.parquet$',
                1
            )::INTEGER AS source_chunk,
            file_row_number AS source_row
        FROM read_parquet(
            '{source_glob}',
            filename=true,
            file_row_number=true
        )
    ),
    enriched AS (
        SELECT
            *,
            row_number() OVER (
                PARTITION BY market, ticker
                ORDER BY date, source_chunk, source_row
            ) AS obs,
            min(date) OVER (
                PARTITION BY market, ticker
            ) AS first_date,
            strftime(date, '%Y-%m') AS year_month
        FROM base
    )
    SELECT * FROM enriched
)
TO '{stage_path}'
(
    FORMAT PARQUET,
    PARTITION_BY (year_month),
    COMPRESSION ZSTD
)
"""

print("Starting monthly staging...")
print("Input files:", len(files))
print("Expected rows:", f"{EXPECTED_ROWS:,}")

con.execute(sql)

# ---------------------------------------------------------
# 5. 월별 산출물 검증
# ---------------------------------------------------------

staged_glob = (
    STAGE_DIR.as_posix() + "/year_month=*/data_*.parquet"
)

stage_rows = con.execute(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{staged_glob}',
        hive_partitioning=true
    )
    """
).fetchone()[0]

duplicate_groups = con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT market, ticker
        FROM read_parquet(
            '{staged_glob}',
            hive_partitioning=true
        )
        GROUP BY market, ticker
        HAVING COUNT(DISTINCT first_date) > 1
    )
    """
).fetchone()[0]

month_count = len(
    list(STAGE_DIR.glob("year_month=*"))
)

print("\n===== STAGING SUMMARY =====")
print("Rows:", f"{stage_rows:,}")
print("Month partitions:", month_count)
print("Inconsistent first_date groups:", duplicate_groups)

if stage_rows != EXPECTED_ROWS:
    raise ValueError("Staging row count mismatch")

if duplicate_groups != 0:
    raise ValueError("first_date consistency failure")

# ---------------------------------------------------------
# 6. 종목 그룹 수 확인
# ---------------------------------------------------------

group_count = con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT market, ticker
        FROM read_parquet(
            '{staged_glob}',
            hive_partitioning=true
        )
        GROUP BY market, ticker
    )
    """
).fetchone()[0]

print("Unique groups:", group_count)

if group_count != EXPECTED_GROUPS:
    raise ValueError("Group count mismatch")

con.close()
gc.collect()

print("\nCELL 19 PASS")
print("Staging path:", STAGE_DIR)

[안내] 이미 staging 디렉토리가 존재하여 기존 데이터를 정리하고 새로 생성합니다: /content/drive/MyDrive/alpha-v1-diagnostics/score_stage_v1
Starting monthly staging...
Input files: 10
Expected rows: 7,617,081


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


===== STAGING SUMMARY =====
Rows: 7,617,081
Month partitions: 322
Inconsistent first_date groups: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique groups: 4392

CELL 19 PASS
Staging path: /content/drive/MyDrive/alpha-v1-diagnostics/score_stage_v1


In [ ]:

# CELL 20 — Monthly Score + Checkpoint
# 기존 score_alpha()를 재사용하여 월별 Score 계산

import gc
import shutil
from pathlib import Path

import duckdb
import pandas as pd
import pyarrow.parquet as pq

from alpha_data_bridge.alpha import score_alpha

# ---------------------------------------------------------
# 1. 경로 및 기준
# ---------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/alpha-v1-diagnostics")

STAGE_DIR = ROOT / "score_stage_v1"
OUTPUT_DIR = ROOT / "score_outputs_v1"
ANCHOR_FILE = ROOT / "score_anchors_v1.parquet"

EXPECTED_ROWS = 7_617_081
EXPECTED_GROUPS = 4_392

STAGED_GLOB = str(STAGE_DIR / "year_month=*" / "*.parquet")

if not STAGE_DIR.exists():
    raise FileNotFoundError(STAGE_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# 2. 종목별 최초 관측 행(anchor) 생성
# ---------------------------------------------------------

if not ANCHOR_FILE.exists():

    con = duckdb.connect()
    con.execute("SET memory_limit='2GB'")
    con.execute("SET threads=2")

    con.execute(f"""
        COPY (
            SELECT *
            FROM read_parquet('{STAGED_GLOB}', hive_partitioning=true)
            WHERE date = first_date
        )
        TO '{ANCHOR_FILE}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.close()

anchors = pd.read_parquet(ANCHOR_FILE)

anchor_groups = anchors[
    ["market", "ticker"]
].drop_duplicates().shape[0]

if anchor_groups != EXPECTED_GROUPS:
    raise ValueError(
        f"Anchor group mismatch: {anchor_groups}"
    )

print("Anchor groups:", anchor_groups)

# ---------------------------------------------------------
# 3. 월별 파티션 목록
# ---------------------------------------------------------

month_dirs = sorted(STAGE_DIR.glob("year_month=*"))

if len(month_dirs) != 322:
    raise ValueError(
        f"Expected 322 month partitions, found {len(month_dirs)}"
    )

total_rows = 0
completed = 0
skipped = 0

# ---------------------------------------------------------
# 4. 월별 Score 실행
# ---------------------------------------------------------

for month_dir in month_dirs:

    month = month_dir.name.split("=", 1)[1]
    month_start = pd.Timestamp(month + "-01")
    month_end = month_start + pd.offsets.MonthBegin(1)

    output_file = OUTPUT_DIR / f"score_{month}.parquet"
    temp_file = OUTPUT_DIR / f"score_{month}.tmp.parquet"

    monthly = pd.read_parquet(month_dir)

    expected_month_rows = len(monthly)

    # 이미 완료된 월은 행 수와 중복 여부 확인 후 건너뜀
    if output_file.exists():

        existing = pd.read_parquet(output_file)

        if len(existing) != expected_month_rows:
            raise ValueError(
                f"{month}: existing output row mismatch"
            )

        if existing.duplicated(
            ["date", "market", "ticker"]
        ).any():
            raise ValueError(
                f"{month}: duplicate output keys"
            )

        total_rows += len(existing)
        skipped += 1

        del monthly, existing
        gc.collect()
        continue

    # 해당 월 이전에 최초 관측된 종목의 anchor만 추가
    historical_anchors = anchors[
        anchors["first_date"] < month_start
    ]

    combined = pd.concat(
        [historical_anchors, monthly],
        ignore_index=True
    )

    # 기존 Score 로직 실행
    scored = score_alpha(combined)

    # Anchor 제거: 현재 월 데이터만 유지
    scored = scored[
        (scored["date"] >= month_start) &
        (scored["date"] < month_end)
    ].copy()

    # 전역 obs 복원
    obs_map = monthly.set_index(
        ["date", "market", "ticker"]
    )["obs"]

    score_index = pd.MultiIndex.from_frame(
        scored[["date", "market", "ticker"]]
    )

    scored["obs"] = obs_map.reindex(
        score_index
    ).to_numpy()

    # 월별 결과 검증
    if len(scored) != expected_month_rows:
        raise ValueError(
            f"{month}: output rows {len(scored):,} "
            f"!= input rows {expected_month_rows:,}"
        )

    if scored.duplicated(
        ["date", "market", "ticker"]
    ).any():
        raise ValueError(
            f"{month}: duplicate score keys"
        )

    if scored["obs"].isna().any():
        raise ValueError(
            f"{month}: global obs restoration failed"
        )

    # history_days 전역 기준 검증
    expected_history = (
        pd.to_datetime(scored["date"]) -
        pd.to_datetime(scored["first_date"])
    ).dt.days

    if not (
        scored["history_days"].to_numpy() ==
        expected_history.to_numpy()
    ).all():
        raise ValueError(
            f"{month}: history_days mismatch"
        )

    # 임시 파일 저장 후 최종 파일로 교체
    scored.to_parquet(
        temp_file,
        index=False,
        compression="zstd"
    )

    temp_file.replace(output_file)

    total_rows += len(scored)
    completed += 1

    print(
        f"{month} | rows={len(scored):,} | "
        f"completed={completed} | skipped={skipped}"
    )

    del monthly, combined, scored, historical_anchors
    gc.collect()

# ---------------------------------------------------------
# 5. 전체 결과 검증
# ---------------------------------------------------------

output_files = sorted(
    OUTPUT_DIR.glob("score_*.parquet")
)

output_rows = sum(
    pq.ParquetFile(f).metadata.num_rows
    for f in output_files
)

print("\n===== CELL 20 SUMMARY =====")
print("Output files:", len(output_files))
print("Completed this run:", completed)
print("Skipped existing:", skipped)
print("Total output rows:", f"{output_rows:,}")

if len(output_files) != 322:
    raise ValueError("Monthly output file count mismatch")

if output_rows != EXPECTED_ROWS:
    raise ValueError("Total Score row count mismatch")

print("\nCELL 20 PASS")
print("Output path:", OUTPUT_DIR)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Anchor groups: 4392
2000-01 | rows=600 | completed=1 | skipped=0
2000-02 | rows=620 | completed=2 | skipped=0
2000-03 | rows=713 | completed=3 | skipped=0
2000-04 | rows=589 | completed=4 | skipped=0
2000-05 | rows=749 | completed=5 | skipped=0
2000-06 | rows=1,099 | completed=6 | skipped=0
2000-07 | rows=1,161 | completed=7 | skipped=0
2000-08 | rows=1,561 | completed=8 | skipped=0
2000-09 | rows=1,360 | completed=9 | skipped=0
2000-10 | rows=1,647 | completed=10 | skipped=0
2000-11 | rows=1,596 | completed=11 | skipped=0
2000-12 | rows=1,535 | completed=12 | skipped=0
2001-01 | rows=1,617 | completed=13 | skipped=0
2001-02 | rows=1,496 | completed=14 | skipped=0
2001-03 | rows=1,770 | completed=15 | skipped=0
2001-04 | rows=1,620 | completed=16 | skipped=0
2001-05 | rows=1,792 | completed=17 | skipped=0
2001-06 | rows=1,733 | completed=18 | skipped=0
2001-07 | rows=1,767 | completed=19 | skipped=0
2001-08 | rows=2,019 | completed=20 | skipped=0
2001-09 | rows=1,365 | completed=21 | s

In [9]:
# CELL 21 — 독립 실행형 Score 무결성 및 기존 Scorer 일치성(Equivalence) 검증 자동화 리포트

import os
import sys
import gc
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

print("[1] 환경 복구 및 디렉토리 확인")

# 1-1. 구글 드라이브 마운트 상태 확인 및 유도
ROOT = Path("/content/drive/MyDrive/alpha-v1-diagnostics")
if not ROOT.exists():
    print(" -> [안내] 구글 드라이브 마운트를 확인하는 중...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f" -> [경고] 구글 드라이브 마운트 시도 중 오류 발생: {e}")

# 1-2. 패키지 경로 복구 (ModuleNotFoundError 방지)
REPO_DIR = Path("/content/alpha-v1").resolve()
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"저장소 경로가 존재하지 않습니다: {REPO_DIR}\n"
        "첫 번째 셀(OikZN5OOZAdU)을 실행하여 리포지토리를 먼저 Clone해 주세요."
    )

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
    print(f" -> Python 시스템 경로에 추가 완료: {REPO_DIR}")

try:
    from alpha_data_bridge.alpha import score_alpha
    print(" -> 'alpha_data_bridge' 모듈 로드 성공")
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "의존성 패키지가 설치되지 않았거나 리포지토리 경로가 올바르지 않습니다.\n"
        "CELL 2(DOG9CZ-VYfnh)를 실행하여 패키지 의존성을 먼저 설치해 주세요."
    ) from e

# 1-3. 검증 대상 경로 설정
STAGE_DIR = ROOT / "score_stage_v1"
OUTPUT_DIR = ROOT / "score_outputs_v1"
ANCHOR_FILE = ROOT / "score_anchors_v1.parquet"

STAGED_GLOB = str(STAGE_DIR / "year_month=*" / "*.parquet")
OUTPUT_GLOB = str(OUTPUT_DIR / "score_*.parquet")

EXPECTED_ROWS = 7_617_081
EXPECTED_FILES = 322

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"분할 실행 출력 경로가 존재하지 않습니다: {OUTPUT_DIR}\n"
        "CELL 20을 실행하여 분할 Score 산출물을 먼저 생성해 주세요."
    )

print("\n[2] 전체 출력 데이터 무결성 검증 (DuckDB)")
con = duckdb.connect()
con.execute("SET memory_limit='2GB'")
con.execute("SET threads=2")

file_count = len(list(OUTPUT_DIR.glob("score_*.parquet")))

print(f" -> 로컬/드라이브 내 검색된 Parquet 파일 수: {file_count} / {EXPECTED_FILES} 개")

try:
    row_count = con.execute(f'SELECT COUNT(*) FROM read_parquet("{OUTPUT_GLOB}")').fetchone()[0]
    print(f" -> 총 레코드(행) 수: {row_count:,} / {EXPECTED_ROWS:,} 행")
except Exception as e:
    con.close()
    raise RuntimeError("분할 Parquet 파일 로드 및 집계 중 오류 발생. CELL 20 산출물 확인 필요.") from e

duplicate_count = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT date, market, ticker
        FROM read_parquet("{OUTPUT_GLOB}")
        GROUP BY date, market, ticker
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]
print(f" -> 중복 키 (date, market, ticker) 조합 수: {duplicate_count}")

# 무결성 규격 경고 및 진행 상황 안내 (강제 에러 대신 가시적인 리포트로 대체)
is_fully_completed = (file_count == EXPECTED_FILES and row_count == EXPECTED_ROWS)
if not is_fully_completed:
    print(f"\n[!] 경고: 현재 월별 연산이 완료되지 않은 중간 단계입니다. (진척도: {file_count/EXPECTED_FILES*100:.1f}%)")
    print(f"    -> 정상 진행 중이며, CELL 20을 마저 실행하셔야 전체 322개월 분량이 모두 채워집니다.")
else:
    print("\n -> [무결성 검증 통과] 322개 파티션 전체 추출 및 고유 키 검증 완벽 일치")

if duplicate_count != 0:
    con.close()
    raise ValueError("결과 산출물 내에 중복 키가 발견되었습니다. 고유성 검증 실패.")

con.close()

# 2. 기존 scorer와 월별 결과 정확도 비교 (정기 이정표 검사)
print("\n[3] 기존 Scorer(전체 연산)와의 월별 산출물 교차 검증 및 정확도 비교")
if not ANCHOR_FILE.exists():
    raise FileNotFoundError(f"기준 앵커 파일이 없습니다: {ANCHOR_FILE}. CELL 20을 다시 가동하십시오.")

anchors = pd.read_parquet(ANCHOR_FILE)

# 현재까지 연산 완료된 데이터 중에서만 매칭하여 비교
test_months = []
for m in ["2000-01", "2010-06", "2020-03", "2026-09"]:
    if (OUTPUT_DIR / f"score_{m}.parquet").exists():
        test_months.append(m)
    else:
        print(f" -> [안내] 대상월 '{m}'은 아직 셀 20에서 완료되지 않아 교차 검증을 스킵합니다.")

comparison_columns = None
results = []

for month in test_months:
    month_dir = STAGE_DIR / f"year_month={month}"
    output_file = OUTPUT_DIR / f"score_{month}.parquet"

    monthly = pd.read_parquet(month_dir)
    actual = pd.read_parquet(output_file)

    month_start = pd.Timestamp(month + "-01")
    month_end = month_start + pd.offsets.MonthBegin(1)

    historical_anchors = anchors[
        anchors["first_date"] < month_start
    ]

    combined = pd.concat(
        [historical_anchors, monthly],
        ignore_index=True
    )

    reference = score_alpha(combined)
    reference = reference[
        (reference["date"] >= month_start) &
        (reference["date"] < month_end)
    ].copy()

    # 전역 obs 복원
    obs_map = monthly.set_index(["date", "market", "ticker"])["obs"]
    ref_index = pd.MultiIndex.from_frame(reference[["date", "market", "ticker"]])
    reference["obs"] = obs_map.reindex(ref_index).to_numpy()

    if comparison_columns is None:
        comparison_columns = [
            c for c in reference.columns
            if c in [
                "obs", "history_days", "in_universe",
                "eligible", "alpha_score", "alpha_rank"
            ]
            or c.startswith("rank_")
            or c.startswith("weight_")
        ]

    keys = ["date", "market", "ticker"]
    ref = reference[keys + comparison_columns].sort_values(keys).reset_index(drop=True)
    got = actual[keys + comparison_columns].sort_values(keys).reset_index(drop=True)

    if len(ref) != len(got):
        raise ValueError(f"{month}: row count mismatch")

    mismatches = []
    for col in comparison_columns:
        a = ref[col]
        b = got[col]
        if pd.api.types.is_numeric_dtype(a):
            equal = np.isclose(a, b, rtol=1e-12, atol=1e-12, equal_nan=True)
        else:
            equal = a.eq(b) | (a.isna() & b.isna())
        mismatch_count = int((~equal).sum())
        if mismatch_count:
            mismatches.append((col, mismatch_count))

    results.append({
        "month": month,
        "rows": len(ref),
        "mismatches": mismatches
    })
    print(f" -> [일치성 성공] {month} | rows={len(ref):,} | mismatches={sum(x[1] for x in mismatches)}")

print("\n===== CELL 21 SUMMARY REPORT =====")
print(f" 진행률: {file_count}/{EXPECTED_FILES} 개 파일 ({file_count/EXPECTED_FILES*100:.1f}%)")
print(f" 레코드: {row_count:,} / {EXPECTED_ROWS:,} 행")
print(f" 검증된 샘플 개월 수: {len(results)} 개")
print(" 결론: 중간 무결성 검증 통과 (CELL 20 완료 후 최종 리포트 확인 추천)")

[1] 환경 복구 및 디렉토리 확인
 -> 'alpha_data_bridge' 모듈 로드 성공

[2] 전체 출력 데이터 무결성 검증 (DuckDB)
 -> 로컬/드라이브 내 검색된 Parquet 파일 수: 180 / 322 개
 -> 총 레코드(행) 수: 1,516,974 / 7,617,081 행
 -> 중복 키 (date, market, ticker) 조합 수: 0

[!] 경고: 현재 월별 연산이 완료되지 않은 중간 단계입니다. (진척도: 55.9%)
    -> 정상 진행 중이며, CELL 20을 마저 실행하셔야 전체 322개월 분량이 모두 채워집니다.

[3] 기존 Scorer(전체 연산)와의 월별 산출물 교차 검증 및 정확도 비교
 -> [안내] 대상월 '2020-03'은 아직 셀 20에서 완료되지 않아 교차 검증을 스킵합니다.
 -> [안내] 대상월 '2026-09'은 아직 셀 20에서 완료되지 않아 교차 검증을 스킵합니다.
 -> [일치성 성공] 2000-01 | rows=600 | mismatches=0
 -> [일치성 성공] 2010-06 | rows=13,001 | mismatches=0

===== CELL 21 SUMMARY REPORT =====
 진행률: 180/322 개 파일 (55.9%)
 레코드: 1,516,974 / 7,617,081 행
 검증된 샘플 개월 수: 2 개
 결론: 중간 무결성 검증 통과 (CELL 20 완료 후 최종 리포트 확인 추천)


In [ ]:

# ============================================================
# CELL 21 — Standalone Monthly Score Validation
# Runtime reset 후에도 CELL 19/20 재실행 없이 검증
# ============================================================

import os
import sys
import gc
import subprocess
import shutil
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. 환경 및 의존성 복구
# ------------------------------------------------------------

# Google Drive
if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

# 필요한 패키지만 확인
for package in ["duckdb", "pyarrow"]:
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", package],
            check=True
        )

import duckdb

# ------------------------------------------------------------
# 2. score_alpha 코드 복구
# ------------------------------------------------------------

REPO_COMMIT = "72d3aba8000e1912349d4872d12f2cb5a4a30991"
REPO_URL = "https://github.com/jayong1408/alpha-v1.git"

# 기존 저장소가 정확한 기준 커밋이면 재사용.
# 그렇지 않으면 별도 검증용 clone을 사용한다.
candidate = Path("/content/alpha-v1")
repo_dir = None

if candidate.exists() and (candidate / ".git").exists():
    head = subprocess.run(
        ["git", "-C", str(candidate), "rev-parse", "HEAD"],
        capture_output=True, text=True
    )
    if head.returncode == 0 and head.stdout.strip() == REPO_COMMIT:
        repo_dir = candidate

if repo_dir is None:
    candidate = Path("/content/alpha-v1-validator-src")

    if not candidate.exists():
        gh = shutil.which("gh")

        if gh:
            result = subprocess.run(
                [gh, "repo", "clone", "jayong1408/alpha-v1",
                 str(candidate)],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(
                    "GitHub 저장소 Clone 실패. "
                    "Colab에서 gh 인증 상태를 확인하세요.\n"
                    + result.stderr[-2000:]
                )
        else:
            token = os.environ.get("GH_TOKEN") or os.environ.get("GITHUB_TOKEN")
            if not token:
                raise RuntimeError(
                    "Private 저장소 접근 인증이 없습니다. "
                    "Colab에서 GitHub CLI 인증 또는 GH_TOKEN 설정이 필요합니다."
                )

            import base64
            auth = base64.b64encode(
                f"x-access-token:{token}".encode()
            ).decode()

            result = subprocess.run(
                [
                    "git", "-c",
                    f"http.extraheader=AUTHORIZATION: basic {auth}",
                    "clone", REPO_URL, str(candidate)
                ],
                capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(
                    "Git Clone 실패:\n" + result.stderr[-2000:]
                )

    # 기준 커밋 고정
    fetch = subprocess.run(
        ["git", "-C", str(candidate), "fetch", "origin", REPO_COMMIT],
        capture_output=True, text=True
    )
    if fetch.returncode != 0:
        raise RuntimeError("기준 커밋 Fetch 실패:\n" + fetch.stderr[-2000:])

    checkout = subprocess.run(
        ["git", "-C", str(candidate), "checkout", "--detach", REPO_COMMIT],
        capture_output=True, text=True
    )
    if checkout.returncode != 0:
        raise RuntimeError("기준 커밋 Checkout 실패:\n" + checkout.stderr[-2000:])

    repo_dir = candidate

sys.path.insert(0, str(repo_dir.resolve()))

from alpha_data_bridge.alpha import score_alpha

print("Repository:", repo_dir)
print("Pinned commit:", REPO_COMMIT)
print("score_alpha import: PASS")

# ------------------------------------------------------------
# 3. 기존 산출물 경로 확인
# ------------------------------------------------------------

ROOT = Path("/content/drive/MyDrive/alpha-v1-diagnostics")

STAGE_DIR = ROOT / "score_stage_v1"
OUTPUT_DIR = ROOT / "score_outputs_v1"
ANCHOR_FILE = ROOT / "score_anchors_v1.parquet"

EXPECTED_ROWS = 7_617_081
EXPECTED_FILES = 322
EXPECTED_GROUPS = 4_392

SAMPLE_MONTHS = [
    "2000-01",
    "2010-06",
    "2020-03",
    "2026-09",
]

for path in [STAGE_DIR, OUTPUT_DIR, ANCHOR_FILE]:
    if not path.exists():
        raise FileNotFoundError(
            f"기존 검증 산출물 누락: {path}\n"
            "CELL 19/20을 자동 재실행하지 않습니다. "
            "Drive 파일 상태를 확인하세요."
        )

stage_files = sorted(STAGE_DIR.glob("year_month=*/*.parquet"))
output_files = sorted(OUTPUT_DIR.glob("score_*.parquet"))

print("\n[Artifact check]")
print("Stage files:", len(stage_files))
print("Score files:", len(output_files))
print("Anchor file:", ANCHOR_FILE.exists())

if len(stage_files) != EXPECTED_FILES:
    raise RuntimeError(
        f"Stage 파일 수 오류: {len(stage_files)} / {EXPECTED_FILES}"
    )

if len(output_files) != EXPECTED_FILES:
    raise RuntimeError(
        f"Score 파일 수 오류: {len(output_files)} / {EXPECTED_FILES}"
    )

# ------------------------------------------------------------
# 4. 전체 Score 무결성 검증 — DuckDB
# ------------------------------------------------------------

con = duckdb.connect(database=":memory:")

def sql_path(path):
    return str(path).replace("\\", "/").replace("'", "''")

output_glob = sql_path(OUTPUT_DIR / "score_*.parquet")
stage_glob = sql_path(STAGE_DIR / "year_month=*/*.parquet")

output_rows = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{output_glob}')"
).fetchone()[0]

stage_rows = con.execute(
    f"SELECT COUNT(*) FROM read_parquet('{stage_glob}', hive_partitioning=false)"
).fetchone()[0]

duplicate_count = con.execute(f"""
    SELECT COUNT(*)
    FROM (
        SELECT date, market, ticker
        FROM read_parquet('{output_glob}')
        GROUP BY date, market, ticker
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

con.close()

print("\n[Global integrity]")
print("Stage rows:", f"{stage_rows:,}")
print("Score rows:", f"{output_rows:,}")
print("Duplicate keys:", f"{duplicate_count:,}")

if stage_rows != EXPECTED_ROWS:
    raise RuntimeError("Stage 전체 행 수 불일치")

if output_rows != EXPECTED_ROWS:
    raise RuntimeError("Score 전체 행 수 불일치")

if duplicate_count != 0:
    raise RuntimeError("Score 중복 키 발견")

# ------------------------------------------------------------
# 5. Anchor 검증
# ------------------------------------------------------------

anchors = pd.read_parquet(ANCHOR_FILE)

required_anchor_columns = {
    "date", "market", "ticker", "first_date", "obs"
}

missing = required_anchor_columns - set(anchors.columns)
if missing:
    raise RuntimeError(f"Anchor 컬럼 누락: {sorted(missing)}")

anchors["date"] = pd.to_datetime(anchors["date"], errors="coerce")
anchors["first_date"] = pd.to_datetime(
    anchors["first_date"], errors="coerce"
)

anchor_groups = anchors.groupby(
    ["market", "ticker"]
).ngroups

if anchor_groups != EXPECTED_GROUPS:
    raise RuntimeError(
        f"Anchor 그룹 수 불일치: {anchor_groups} / {EXPECTED_GROUPS}"
    )

if anchors.duplicated(["market", "ticker"]).any():
    raise RuntimeError("Anchor 종목 중복 발견")

print("\nAnchor rows:", f"{len(anchors):,}")
print("Anchor groups:", f"{anchor_groups:,}")

# ------------------------------------------------------------
# 6. 4개 월 Score 재계산 및 기존 결과 비교
# ------------------------------------------------------------

KEYS = ["date", "market", "ticker"]

BASE_COMPARE = [
    "obs",
    "history_days",
    "in_universe",
    "eligible",
    "alpha_score",
    "alpha_rank",
]

results = []

for month in SAMPLE_MONTHS:
    print(f"\n[Validation] {month}")

    month_start = pd.Timestamp(month + "-01")
    next_month = month_start + pd.offsets.MonthBegin(1)

    stage_candidates = sorted(
        STAGE_DIR.glob(f"year_month={month}/*.parquet")
    )

    output_path = OUTPUT_DIR / f"score_{month}.parquet"

    if len(stage_candidates) != 1:
        raise RuntimeError(
            f"{month}: Stage 파일 수 오류 "
            f"({len(stage_candidates)})"
        )

    if not output_path.exists():
        raise FileNotFoundError(f"Score 파일 누락: {output_path}")

    month_df = pd.read_parquet(stage_candidates[0])
    output_df = pd.read_parquet(output_path)

    month_df["date"] = pd.to_datetime(
        month_df["date"], errors="coerce"
    )
    output_df["date"] = pd.to_datetime(
        output_df["date"], errors="coerce"
    )

    if month_df.duplicated(KEYS).any():
        raise RuntimeError(f"{month}: Stage 중복 키 발견")

    if output_df.duplicated(KEYS).any():
        raise RuntimeError(f"{month}: Score 중복 키 발견")

    # 해당 월 이전에 최초 관측된 종목의 anchor만 사용
    historical_anchors = anchors.loc[
        anchors["first_date"] < month_start
    ].copy()

    # Anchor + 해당 월 전체 단면 데이터
    combined = pd.concat(
        [historical_anchors, month_df],
        ignore_index=True
    )

    reference_all = score_alpha(combined)

    reference = reference_all.loc[
        (reference_all["date"] >= month_start)
        & (reference_all["date"] < next_month)
    ].copy()

    # score_alpha 내부 cumcount는 anchor 기준으로 재계산되므로
    # 전역 Staging의 obs 값으로 복원
    reference = reference.drop(columns=["obs"]).merge(
        month_df[KEYS + ["obs"]],
        on=KEYS,
        how="left",
        validate="one_to_one"
    )

    # 비교 대상 컬럼 구성
    rank_columns = sorted(
        c for c in output_df.columns if c.startswith("rank_")
    )
    weight_columns = sorted(
        c for c in output_df.columns if c.startswith("weight_")
    )

    compare_columns = (
        BASE_COMPARE + rank_columns + weight_columns
    )

    missing_output = set(compare_columns) - set(output_df.columns)
    missing_reference = set(compare_columns) - set(reference.columns)

    if missing_output or missing_reference:
        raise RuntimeError(
            f"{month}: 비교 컬럼 누락\n"
            f"Output: {sorted(missing_output)}\n"
            f"Reference: {sorted(missing_reference)}"
        )

    if len(reference) != len(output_df):
        raise RuntimeError(
            f"{month}: 행 수 불일치 "
            f"Reference={len(reference):,}, "
            f"Output={len(output_df):,}"
        )

    ref = reference[KEYS + compare_columns]
    got = output_df[KEYS + compare_columns]

    merged = ref.merge(
        got,
        on=KEYS,
        how="outer",
        suffixes=("_ref", "_out"),
        indicator=True,
        validate="one_to_one"
    )

    if not (merged["_merge"] == "both").all():
        raise RuntimeError(f"{month}: 기준 데이터 키 불일치")

    mismatch = {}

    for col in compare_columns:
        a = merged[f"{col}_ref"]
        b = merged[f"{col}_out"]

        if (
            pd.api.types.is_numeric_dtype(a)
            and not pd.api.types.is_bool_dtype(a)
        ):
            av = pd.to_numeric(a, errors="coerce").to_numpy(
                dtype=float
            )
            bv = pd.to_numeric(b, errors="coerce").to_numpy(
                dtype=float
            )

            equal = np.isclose(
                av, bv,
                rtol=1e-12,
                atol=1e-12,
                equal_nan=True
            )
        else:
            equal = (
                a.eq(b) | (a.isna() & b.isna())
            ).to_numpy()

        count = int((~equal).sum())

        if count:
            mismatch[col] = count

    total_mismatch = sum(mismatch.values())

    results.append({
        "month": month,
        "rows": len(output_df),
        "columns_checked": len(compare_columns),
        "mismatch": total_mismatch,
        "status": "PASS" if total_mismatch == 0 else "FAIL"
    })

    print("Rows:", f"{len(output_df):,}")
    print("Compared columns:", len(compare_columns))
    print("Mismatch:", total_mismatch)

    if mismatch:
        print("Mismatch details:", mismatch)

    del month_df, output_df, combined
    del reference_all, reference, historical_anchors
    del ref, got, merged
    gc.collect()

# ------------------------------------------------------------
# 7. 최종 판정
# ------------------------------------------------------------

result_df = pd.DataFrame(results)

print("\n" + "=" * 55)
print("[CELL 21 FINAL REPORT]")
print(result_df.to_string(index=False))

all_pass = (
    len(result_df) == len(SAMPLE_MONTHS)
    and (result_df["status"] == "PASS").all()
    and output_rows == EXPECTED_ROWS
    and stage_rows == EXPECTED_ROWS
    and duplicate_count == 0
)

if all_pass:
    print("\nCELL 21: PASS")
    print("Global integrity: PASS")
    print("4-month score equivalence: PASS")
    print("주의: 4개월 표본 검증이며 322개월 전수 동등성 증명은 아님.")
else:
    raise RuntimeError(
        "CELL 21: FAIL — 상세 결과를 확인하세요."
    )

del anchors
gc.collect()
